In [ ]:
NAME = ""
ID = ""

---

# Beautiful Soup

Beautiful Soup is a Python library for pulling data out of HTML and XML files. It works with your favorite parser to provide idiomatic ways of navigating, searching, and modifying the parse tree. Most content in this chapter is from [here](https://www.crummy.com/software/BeautifulSoup/bs4/doc/).

An [Real-time HTML Editor](https://htmledit.squarefree.com/) may help you to visualize and understand html file.

## 1. Quick Start

Here’s an HTML document which will be used as an example throughout this document. It’s part of a story from Alice in Wonderland:

In [ ]:
html_doc = """<html><head><title>The Dormouse's story</title></head>
<body>
<p class="title"><b>The Dormouse's story</b></p>

<p class="story">Once upon a time there were three little sisters; and their names were
<a href="http://example.com/elsie" class="sister" id="link1">Elsie</a>,
<a href="http://example.com/lacie" class="sister" id="link2">Lacie</a> and
<a href="http://example.com/tillie" class="sister" id="link3">Tillie</a>;
and they lived at the bottom of a well.</p>

<p class="story">...</p>
</body>
</html>
"""

In [ ]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(html_doc, 'html.parser')

print(soup.prettify())

The `prettify()` method will turn a Beautiful Soup parse tree into a nicely formatted Unicode string, with a separate line for each tag and each string.

In [ ]:
print(soup.title)
# <title>The Dormouse's story</title>

print(soup.title.name)
# u'title'


print(soup.title.string)
# u'The Dormouse's story'


print(soup.title.parent.name)
# u'head'

In [ ]:
print(soup.p)
# <p class="title"><b>The Dormouse's story</b></p>

print(soup.p['class'])
# u'title'

In [ ]:
print(soup.a)
# <a class="sister" href="http://example.com/elsie" id="link1">Elsie</a>

print(soup.find_all('a'))
# [<a class="sister" href="http://example.com/elsie" id="link1">Elsie</a>,
#  <a class="sister" href="http://example.com/lacie" id="link2">Lacie</a>,
#  <a class="sister" href="http://example.com/tillie" id="link3">Tillie</a>]

print(soup.find(id="link3"))
# <a class="sister" href="http://example.com/tillie" id="link3">Tillie</a>

Extract all the URLs found within a page’s <a> tags:

In [ ]:
for link in soup.find_all('a'):
    print(link.get('href'))
#     print(link['href'])

If you only want the human-readable text inside a document or tag, you can use the `get_text()` method. It returns all the text:

In [ ]:
print(soup.get_text())

## 2.Making the soup

To parse a document, pass it into the BeautifulSoup constructor to create a  BeautifulSoup object, which represents the document as a nested data structure. You can pass in a string or an open filehandle:

In [ ]:
import bs4

with open("Files/index.html") as fp:
    soup = BeautifulSoup(fp, 'html.parser')
    print(soup)

In [ ]:
soup = BeautifulSoup("<html>a web page</html>", 'html.parser')
print(soup)
print(BeautifulSoup("<html><head></head><body>Sacr&eacute; bleu!</body></html>", "html.parser"))

Beautiful Soup transforms a complex HTML document into a complex tree of Python objects. But you’ll only ever have to deal with about four kinds of objects: `Tag`, `NavigableString`, `BeautifulSoup`, and `Comment`.

### Tag

A Tag object corresponds to an XML or HTML tag in the original document. Tags have a lot of attributes and methods. For now, the most important features of a tag are its name and attributes.

In [ ]:
soup = BeautifulSoup('<b class="boldest">Extremely bold</b>', 'html.parser')
tag = soup.b
print(type(tag))

print(tag.name)

tag.name = "blockquote"
print(tag)

A tag may have any number of attributes. The tag `<b id="boldest">`  has an attribute "id" whose value is "boldest". You can access a tag’s attributes by treating the tag like a dictionary or attrs.

In [ ]:
tag = BeautifulSoup('<b id="boldest">bold</b>', 'html.parser').b
print(tag['id'])
print(tag.attrs)

You can add, remove, and modify a tag’s attributes. Again, this is done by treating the tag as a dictionary:

In [ ]:
tag['id'] = 'verybold'
tag['another-attribute'] = 1
print(tag)
# <b another-attribute="1" id="verybold"></b>

In [ ]:
del tag['id']
del tag['another-attribute']
print(tag)

In [ ]:
tag['id']
# KeyError: 'id'

In [ ]:
print(tag.get('id'))
# None

HTML 4 defines a few attributes that can have multiple values. HTML 5 removes a couple of them, but defines a few more. **The most common multi-valued attribute is `class` (that is, a tag can have more than one CSS class).** Others include rel, rev, accept-charset, headers, and accesskey. Beautiful Soup presents the value(s) of a multi-valued attribute as a list. 

If an attribute looks like it has more than one value, but it’s not a multi-valued attribute as defined by any version of the HTML standard, Beautiful Soup will leave the attribute alone.


In [ ]:
css_soup = BeautifulSoup('<p class="body"></p>', 'html.parser')
print(css_soup.p['class'])
# ['body']

In [ ]:
css_soup = BeautifulSoup('<p class="body strikeout"></p>', 'html.parser')
print(css_soup.p['class'])
# ['body', 'strikeout']

In [ ]:
id_soup = BeautifulSoup('<p id="my id"></p>', 'html.parser')
print(id_soup.p['id'])

### NavigableString

A string corresponds to a bit of text within a tag. Beautiful Soup uses the NavigableString class to contain these bits of text. 

A NavigableString is just like a Python Unicode string, except that it also supports some of the features described in Navigating the tree and Searching the tree. You can convert a NavigableString to a Unicode string with str (in Python 3):

**You can’t edit a string in place, but you can replace one string with another, using `replace_with()`**.

In [ ]:
soup = BeautifulSoup('<b class="boldest">Extremely bold</b>', 'html.parser')
tag = soup.b
print(tag.string)
print(type(tag.string))

In [ ]:
unicode_string = str(tag.string)
unicode_string
print(type(unicode_string))

In [ ]:
tag.string.replace_with("No longer bold")
print(tag)

### BeautifulSoup

The BeautifulSoup object represents the parsed document as a whole. For most purposes, you can treat it as a Tag object. This means it supports most of the methods described in Navigating the tree and Searching the tree.

You can also pass a BeautifulSoup object into one of the methods defined in Modifying the tree, just as you would a Tag. This lets you do things like combine two parsed documents.

### Comments and other special strings

In [ ]:
markup = "<b><!--Hey, buddy. Want to buy a used parser?--></b>"
soup = BeautifulSoup(markup, 'html.parser')
comment = soup.b.string
print(type(comment))
print(comment)

## 3.Navigating the tree

### Going down

####  Navigating using tag names
The simplest way to navigate the parse tree is to say the name of the tag you want. If you want the `<head>` tag, just say `soup.head`.

**Using a tag name as an attribute will give you only the first tag by that name.**
    
If you need to get all the tags, or anything more complicated than the first tag with a certain name, you’ll need to use one of the methods described in Searching the tree, such as `find_all()`.

In [ ]:
html_doc = """<html><head><title>The Dormouse's story</title></head>
<body>
<p class="title"><b>The Dormouse's story</b></p>

<p class="story">Once upon a time there were three little sisters; and their names were
<a href="http://example.com/elsie" class="sister" id="link1">Elsie</a>,
<a href="http://example.com/lacie" class="sister" id="link2">Lacie</a> and
<a href="http://example.com/tillie" class="sister" id="link3">Tillie</a>;
and they lived at the bottom of a well.</p>

<p class="story">...</p>

</body>
</html>
"""

In [ ]:
soup = BeautifulSoup(html_doc, 'html.parser')

print(soup.head)

print(soup.title)

print(soup.body.b)

print(soup.a)

print(soup.find_all('a'))

#### .contents, .children and .descendants
A tag’s children are available in a list called .contents.

In [ ]:
head_tag = soup.head
print(head_tag)
# <head><title>The Dormouse's story</title></head>

In [ ]:
print(head_tag.contents)
# [<title>The Dormouse's story</title>]

In [ ]:
title_tag = head_tag.contents[0]
print(title_tag)
# <title>The Dormouse's story</title>
print(title_tag.contents)

Instead of getting them as a list, you can iterate over a tag’s children using the `.children` generator:

In [ ]:
for child in title_tag.children:
    print(child)

The .contents and .children attributes only consider a tag’s direct children. For instance, the <head> tag has a single direct child–the <title> tag:

In [ ]:
print(len(list(head_tag.children)))
# 1
print(len(list(head_tag.descendants)))
# 2

for child in head_tag.descendants:
    print(child)

#### .string, .strings and stripped_strings

If a tag has only one child, and that child is a NavigableString, the child is made available as `.string`. 

In [ ]:
print(title_tag)
# <title>The Dormouse's story</title>

print(title_tag.string)
# 'The Dormouse's story'

If a tag’s only child is another tag, and that tag has a .string, then the parent tag is considered to have the same `.string` as its child.

In [ ]:
print(head_tag)
# <head><title>The Dormouse's story</title></head>

print(head_tag.string)
# 'The Dormouse's story'

If a tag contains more than one thing, then it’s not clear what `.string` should refer to, so `.string` is defined to be None. But you can use `.strings`. 

In [ ]:
print(soup.html.string)
# None

In [ ]:
for string in soup.html.strings:
    print(string)


These strings tend to have a lot of extra whitespace, which you can remove by using the `.stripped_strings` generator instead.

In [ ]:
for string in soup.html.stripped_strings:
    print(string)

### Going up

####  .parent and .parents

You can access an element’s parent with the `.parent` attribute. The parent of a top-level tag like `<html>` is the `BeautifulSoup object` itself and the `.parent` of a BeautifulSoup object is defined as None. 

In [ ]:
title_tag = soup.title
print(title_tag)
# <title>The Dormouse's story</title>
print(title_tag.parent)
# <head><title>The Dormouse's story</title></head>

In [ ]:
html_tag = soup.html
print(type(html_tag.parent))
# <class 'bs4.BeautifulSoup'>

print(soup.parent)
# None

You can iterate over all of an element’s parents with `.parents`. This example uses `.parents` to travel from an `<a>` tag buried deep within the document, to the very top of the document.

In [ ]:
print("For parents")
link = soup.a
print(link)
# <a class="sister" href="http://example.com/elsie" id="link1">Elsie</a>
for parent in link.parents:
    print(parent.name)

### Going sideways

####  .next_sibling,  .previous_sibling, .next_siblings and .previous_siblings

In [ ]:
sibling_soup = BeautifulSoup("<a><b>text1</b><c>text2</c></b></a>", 'html.parser')
print(sibling_soup.prettify())
#   <a>
#    <b>
#     text1
#    </b>
#    <c>
#     text2
#    </c>
#   </a>

In [ ]:
print(sibling_soup.b.next_sibling)
# <c>text2</c>

In [ ]:
print(sibling_soup.c.previous_sibling)
# <b>text1</b>

In [ ]:
for sibling in soup.a.next_siblings:
    print(repr(sibling))

In [ ]:
for sibling in soup.find(id="link3").previous_siblings:
    print(repr(sibling))

### Going back and forth

####  .next_element,  .previous_element, .next_elements and .previous_elements

The .next_element attribute of a string or tag points to whatever was parsed immediately afterwards. It might be the same as .next_sibling, but it’s usually drastically different.


In [ ]:
last_a_tag = soup.find("a", id="link3")
print(last_a_tag)
# <a class="sister" href="http://example.com/tillie" id="link3">Tillie</a>

In [ ]:
print(last_a_tag.next_sibling)
# ';\nand they lived at the bottom of a well.'

In [ ]:
print(last_a_tag.next_element)
# 'Tillie'

In [ ]:
print(last_a_tag.previous_element)
# ' and\n'

In [ ]:
print(last_a_tag.previous_element.next_element)
# <a class="sister" href="http://example.com/tillie" id="link3">Tillie</a>

In [ ]:
for element in last_a_tag.next_elements:
    print(repr(element))
# 'Tillie'
# ';\nand they lived at the bottom of a well.'
# '\n'
# <p class="story">...</p>
# '...'
# '\n'
# '\n'
# '\n'

## 4. Searching the tree

### Kinds of filters
You can use filter based on a tag’s name, on its attributes, on the text of a string, or on some combination of these.

#### A string
The simplest filter is a string. Pass a string to a search method and Beautiful Soup will perform a match against that exact string. 

This code finds all the `<b>` tags in the document:

In [ ]:
print(soup)
soup.find_all('b')
# [<b>The Dormouse's story</b>]

####  A regular expression

If you pass in a regular expression object, Beautiful Soup will filter against that regular expression using its search() method. 

This code finds all the tags whose names start with the letter “b”; in this case, the `<body>` tag and the `<b>` tag:

In [ ]:
import re
for tag in soup.find_all(re.compile("^b")):
    print(tag.name)
# body
# b

This code finds all the tags whose names contain the letter ‘t’:

In [ ]:
for tag in soup.find_all(re.compile("t")):
    print(tag.name)
# html
# title

#### A list
If you pass in a list, Beautiful Soup will allow a string match against any item in that list. This code finds all the `<a>` tags and all the `<b>` tags

In [ ]:
soup.find_all(["a", "b"])
# [<b>The Dormouse's story</b>,
#  <a class="sister" href="http://example.com/elsie" id="link1">Elsie</a>,
#  <a class="sister" href="http://example.com/lacie" id="link2">Lacie</a>,
#  <a class="sister" href="http://example.com/tillie" id="link3">Tillie</a>]

####  A function

If none of the other matches work for you, define a function that takes an element as its only argument. The function should return True if the argument matches, and False otherwise.

Here’s a function that returns True if a tag defines the “class” attribute but doesn’t define the “id” attribute.

In [ ]:
def has_class_but_no_id(tag):
    return tag.has_attr('class') and not tag.has_attr('id')

print(soup.find_all(has_class_but_no_id))
# [<p class="title"><b>The Dormouse's story</b></p>,
#  <p class="story">Once upon a time there were…bottom of a well.</p>,
#  <p class="story">...</p>]

In [ ]:
def has_class(tag):
    return tag.has_attr('class')

print(soup.find_all(has_class))

In [ ]:
import re
def not_lacie(href):
    return href and not re.compile("lacie").search(href)

print(soup.find_all(href=not_lacie))
# [<a class="sister" href="http://example.com/elsie" id="link1">Elsie</a>,
#  <a class="sister" href="http://example.com/tillie" id="link3">Tillie</a>]

### find_all
Method signature: `find_all(name, attrs, recursive, string, limit, **kwargs)`

The `find_all()` method looks through a tag’s descendants and retrieves all descendants that match your filters.

* The name argument: Consider tags with certain names. 
* The attrs argument: Consider tags with certain attrs value.
* The recursive argument: find_all() examines all the descendants of a tag: its children, its children’s children, and so on. If you only want Beautiful Soup to consider direct children, you can pass in recursive=False.
* The string argument: With string you can search for strings instead of tags. 
* The limit argument: find_all() returns all the tags and strings that match your filters. This can take a while if the document is large. If you don’t need all the results, you can pass in a number for limit.
* The keyword arguments (kwargs): Any argument that’s not recognized will be turned into a filter on one of a tag’s attributes. In particular, you can search by CSS class using the keyword argument class_.

In [ ]:
print(soup.find_all("a"))
# [<a class="sister" href="http://example.com/elsie" id="link1">Elsie</a>,
#  <a class="sister" href="http://example.com/lacie" id="link2">Lacie</a>,
#  <a class="sister" href="http://example.com/tillie" id="link3">Tillie</a>]

In [ ]:
print(soup.find_all("p", "title"))
# [<p class="title"><b>The Dormouse's story</b></p>]

In [ ]:
print(soup.find_all(class_=re.compile("itl")))
# [<p class="title"><b>The Dormouse's story</b></p>]

In [ ]:
def has_six_characters(css_class):
    return css_class is not None and len(css_class) == 6

soup.find_all(class_=has_six_characters)
# [<a class="sister" href="http://example.com/elsie" id="link1">Elsie</a>,
#  <a class="sister" href="http://example.com/lacie" id="link2">Lacie</a>,
#  <a class="sister" href="http://example.com/tillie" id="link3">Tillie</a>]

In [ ]:
print(soup.find_all(id="link2"))
# [<a class="sister" href="http://example.com/lacie" id="link2">Lacie</a>]

In [ ]:
print(soup.html.find_all("title"))
# [<title>The Dormouse's story</title>]

In [ ]:
print(soup.html.find_all("title", recursive=False))
# []

In [ ]:
print(soup.find_all(string=["Tillie", "Elsie", "Lacie"]))
# ['Elsie', 'Lacie', 'Tillie']

In [ ]:
import re
print(soup.find(string=re.compile("sisters")))
# 'Once upon a time there were three little sisters; and their names were\n'

In [ ]:
print(soup.find_all("a", string="Elsie"))
# [<a href="http://example.com/elsie" class="sister" id="link1">Elsie</a>]

In [ ]:
print(soup.find_all("a", limit=2))
# [<a class="sister" href="http://example.com/elsie" id="link1">Elsie</a>,
#  <a class="sister" href="http://example.com/lacie" id="link2">Lacie</a>]


Because `find_all()` is the most popular method in the Beautiful Soup search API, you can use a shortcut for it. If you treat the BeautifulSoup object or a Tag object as though it were a function, then it’s the same as calling `find_all()` on that object. 

In [ ]:
print(soup.find_all("a"))
print(soup("a"))

### More on BeautifulSoup

Read the [online tutorial](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) to learn more about BeautifulSoup.

## Tasks

#### Task 1:

In the code cell below, complete the function `extractLinks` which accepts an HTML string `s` as input. The function finds all hyperlink `<a>` tags that contain an href attribute and returns a list of all URL strings in the order they appear. For example:

- Input: `"<p>Check <a href='[https://google.com](https://google.com)'>Google</a> and <a href='[https://bnbu.edu.cn](https://bnbu.edu.cn)'>BNBU</a>.</p>"`

- Output: `['https://google.com', 'https://bnbu.edu.cn']`

#### Task 2:

In the code cell below, complete the function `extractTextByClass` which accepts an HTML string `s` and a string `class_name` as arguments. The function finds all HTML elements matching the specified `class_name` and returns a list containing the stripped text of each element.

#### Task 3:

In the code cell below, complete the function `findPdfLinks` which accepts an HTML string `s` as input. The function is to locate all `<a>` tags whose href attribute ends with `.pdf` (case-insensitive). Return a list of all matching href PDF strings. For example:

- Input: `"<a href='report.pdf'>Report</a><a href='image.png'>Image</a><a href='SUMMARY.PDF'>Summary</a>"`

- Output: `['report.pdf', 'SUMMARY.PDF']`